In [1]:
from google import genai
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

In [5]:
complaint = """
My headphones arrived yesterday, but the left side is not working.
I want a replacement.
"""

print(complaint)

classification_prompt = f"""
You are an e-commerce customer complaint classification system.

Analyze the following customer complaint.

Complaint:
{complaint}

Classify the complaint into exactly one of these categories:

- Order Issue
- Delivery Issue
- Product Issue
- Return/Replacement
- Refund Issue
- Payment Issue
- Account Issue
- Cancellation
- General Inquiry

Also determine the priority:

- Low
- Medium
- High

Return the result in exactly this format:

Category: <category>
Priority: <priority>
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=classification_prompt
)

print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.



My headphones arrived yesterday, but the left side is not working.
I want a replacement.

Category: Product Issue
Priority: Medium


In [6]:
complaints = [
    "My phone case arrived damaged and I want a replacement.",
    "My order has not arrived even though the delivery date was yesterday.",
    "I was charged twice for the same order.",
    "I want to return the shoes I purchased last week.",
    "I want a refund because the product I received is defective.",
    "I forgot my password and cannot access my account."
]

In [8]:
def classify_complaint(complaint):
    
    prompt = f"""
    You are an e-commerce customer complaint classification system.

    Analyze the following customer complaint.

    Complaint:
    {complaint}

    Choose exactly one category:

    - Order Issue
    - Delivery Issue
    - Product Issue
    - Return/Replacement
    - Refund Issue
    - Payment Issue
    - Account Issue
    - Cancellation
    - General Inquiry

    Choose exactly one priority:

    - Low
    - Medium
    - High

    Return only:

    Category: <category>
    Priority: <priority>
    """

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text

In [19]:
for complaint in complaints:
    print("Complaint:")
    print(complaint)

    print("\nClassification:")
    print(classify_complaint(complaint))


Complaint:
My phone case arrived damaged and I want a replacement.

Classification:
Category: Product Issue
Priority: Medium
Complaint:
My order has not arrived even though the delivery date was yesterday.

Classification:
Category: Delivery Issue
Priority: Medium
Complaint:
I was charged twice for the same order.

Classification:
Category: Payment Issue
Priority: High
Complaint:
I want to return the shoes I purchased last week.

Classification:
Category: Return/Replacement
Priority: Low
Complaint:
I want a refund because the product I received is defective.

Classification:
Category: Product Issue
Priority: High
Complaint:
I forgot my password and cannot access my account.

Classification:
Category: Account Issue
Priority: Medium


In [10]:
allowed_categories = [
    "Order Issue",
    "Delivery Issue",
    "Product Issue",
    "Return/Replacement",
    "Refund Issue",
    "Payment Issue",
    "Account Issue",
    "Cancellation",
    "General Inquiry"
]

allowed_priorities = [
    "Low",
    "Medium",
    "High"
]

In [11]:
def validate_classification(result):
    
    lines = result.splitlines()
    
    category = None
    priority = None
    
    for line in lines:
        if line.startswith("Category:"):
            category = line.replace("Category:", "").strip()
        
        elif line.startswith("Priority:"):
            priority = line.replace("Priority:", "").strip()
    
    category_valid = category in allowed_categories
    priority_valid = priority in allowed_priorities
    
    return {
        "category": category,
        "priority": priority,
        "category_valid": category_valid,
        "priority_valid": priority_valid,
        "valid": category_valid and priority_valid
    }

In [12]:
test_result = """
Category: Product Issue
Priority: Medium
"""

In [13]:
validation = validate_classification(test_result)

print(validation)

{'category': 'Product Issue', 'priority': 'Medium', 'category_valid': True, 'priority_valid': True, 'valid': True}


In [14]:
invalid_result = """
Category: Electronics Problem
Priority: Very Important
"""

In [15]:
validation = validate_classification(invalid_result)

print(validation)

{'category': 'Electronics Problem', 'priority': 'Very Important', 'category_valid': False, 'priority_valid': False, 'valid': False}


In [17]:
def classify_and_validate(complaint):
    
    prompt = f"""
You are an e-commerce customer complaint classification system.

Analyze the following customer complaint.

Complaint:
{complaint}

Choose exactly one category:

- Order Issue
- Delivery Issue
- Product Issue
- Return/Replacement
- Refund Issue
- Payment Issue
- Account Issue
- Cancellation
- General Inquiry

Choose exactly one priority:

- Low
- Medium
- High

Return only:

Category: <category>
Priority: <priority>
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )
    
    result = response.text
    
    validation = validate_classification(result)
    
    return result, validation

In [20]:
complaint = input("Enter your complaint: ")

result, validation = classify_and_validate(complaint)

print("\nGemini Output:")
print(result)

print("\nValidation:")
print(validation)

Enter your complaint:  I recieve the damage headphone



Gemini Output:
Category: Product Issue
Priority: Medium

Validation:
{'category': 'Product Issue', 'priority': 'Medium', 'category_valid': True, 'priority_valid': True, 'valid': True}
